# NB_hospital_daily_summary

Builds `adb_caresync.gold.hospital_daily_summary` (one row per hospital per test day) from silver lab_results, hospitals and calendar, with a day-over-day LAG trend, per the STTM `sttm_silver_to_gold.xlsx`. Full rebuild (CREATE OR REPLACE), LEFT JOINs only.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per test day with test volumes, abnormal/critical rates and day-over-day trend (STTM Silver to Gold v1.0)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                AS summary_date,
    COUNT(*)                                 AS total_tests,
    COUNT(DISTINCT patient_id)               AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  t.hospital_id                                                        AS hospital_id,
  h.hospital_name                                                      AS hospital_name,
  t.summary_date                                                       AS summary_date,
  -- Calendar attributes via optional join; fall back to derived values when the date is outside the calendar range
  COALESCE(c.day_name, date_format(t.summary_date, 'EEEE'))            AS day_name,
  COALESCE(c.is_weekend, dayofweek(t.summary_date) IN (1, 7))          AS is_weekend,
  CAST(t.total_tests AS INT)                                           AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                   AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                   AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                   AS critical_test_count,
  CAST(t.abnormal_test_count * 100.0 / t.total_tests AS DECIMAL(5,2))  AS abnormal_rate_pct,
  CAST(t.critical_test_count * 100.0 / t.total_tests AS DECIMAL(5,2))  AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                  AS prev_day_total_tests,
  CAST(
    CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
         ELSE (t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests
    END AS DECIMAL(6,2))                                               AS test_volume_change_pct
FROM trend t
LEFT JOIN adb_caresync.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))